# Week 14, Day 3: Pseudocode and planning

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w014d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


*Interactive figure on the website: Each step of the plan becomes one or two lines of Python. The plan says what to do; the Python adds only details..*

> **Key idea.** Pseudocode lets you get the **logic** right before the **syntax**. If the plan is wrong, no amount of careful Python will fix it; if the plan is right, the Python is mostly translation.

In [ ]:
def days_in_month(year, month):
    if month == 2:
        leap = year % 4 == 0 and (year % 100 != 0 or year % 400 == 0)
        return 29 if leap else 28
    return 30 if month in (4, 6, 9, 11) else 31

def calendar(year, month, first_weekday):
    """first_weekday: 0 for Monday up to 6 for Sunday."""
    names = ["", "January", "February", "March", "April", "May", "June", "July",
             "August", "September", "October", "November", "December"]
    print(f"{names[month]} {year}".center(20))
    print("Mo Tu We Th Fr Sa Su")
    row = ["  "] * first_weekday
    for day in range(1, days_in_month(year, month) + 1):
        row.append(f"{day:>2}")
        if len(row) == 7:
            print(" ".join(row))
            row = []
    if row:
        print(" ".join(row))

calendar(2026, 10, 3)

> **Tip.** Desk-check with one ordinary example and one edge case. Most plans that survive both are right.

In [ ]:
def to_binary(n):
    if n == 0:
        return "0"
    result = ""
    while n > 0:
        result = str(n % 2) + result   # newest digit goes at the front
        n //= 2
    return result

for n in [6, 0, 1, 255]:
    print(n, to_binary(n), bin(n))

### Exercise: Multiplying by halving and doubling

This method of multiplication was used in ancient Egypt and is often called **Russian peasant multiplication**. It only needs halving, doubling and adding. Translate it into `multiply(a, b)` for whole numbers a, b of at least 0, without using `*` between `a` and `b`.

```
to multiply a by b:
    set total to 0
    while a is more than 0:
        if a is odd, add b to total
        halve a, rounding down
        double b
    the answer is total
```

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    for a, b in [(13, 7), (0, 5), (5, 0), (1, 1), (100, 250), (2026, 31)]:\n        assert multiply(a, b) == a * b, f\"multiply({a}, {b})\"\n    assert \"a * b\" not in source and \"a*b\" not in source")

### Exercise: The sieve of Eratosthenes

Over 2,000 years ago, Eratosthenes of Cyrene described this way to find every prime up to a limit. Translate it into `primes_up_to(n)`.

```
to find the primes up to n:
    make a list of flags for 0 to n, all set to "maybe prime"
    mark 0 and 1 as "not prime"
    for each number p from 2 while p times p is at most n:
        if p is still marked "maybe prime":
            for each multiple of p, starting at p times p, up to n:
                mark it "not prime"
    the answer is every number still marked "maybe prime"
```

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert primes_up_to(30) == [2, 3, 5, 7, 11, 13, 17, 19, 23, 29]\n    assert primes_up_to(1) == [] and primes_up_to(2) == [2]\n    assert len(primes_up_to(10000)) == 1229")

### Exercise: Desk-check this

This plan should return the position of the **last** occurrence of `target` in a list, or -1 if it isn't there. Desk-check it on `[4, 7, 4, 2]` with target 4, and on an empty list. It has one bug. Fix the plan in your head, then write `last_index(items, target)` correctly.

```
to find the last position of target:
    set position to 0
    for each index i of the list, from the start:
        if the item at i is target, set position to i
    the answer is position
```

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert last_index([4, 7, 4, 2], 4) == 2\n    assert last_index([4, 7, 4, 2], 9) == -1\n    assert last_index([], 1) == -1\n    assert last_index([5], 5) == 0")

### Exercise: Refine it yourself

Here's a first version of a plan for `wrap(text, width)`, which wraps text into lines of at most `width` characters, breaking only at spaces (a single word longer than `width` goes on a line by itself). Refine the middle step into detailed steps on paper, desk-check it on `wrap("the quick brown fox", 10)`, then write it. It returns the lines as a list.

```
split the text into words
put the words into lines, starting a new line whenever the next word wouldn't fit
the answer is the lines
```

`wrap("the quick brown fox", 10)` is `["the quick", "brown fox"]`.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert wrap(\"the quick brown fox\", 10) == [\"the quick\", \"brown fox\"]\n    assert wrap(\"the quick brown fox\", 9) == [\"the quick\", \"brown fox\"]\n    assert wrap(\"the quick brown fox\", 8) == [\"the\", \"quick\", \"brown\", \"fox\"]\n    assert wrap(\"a supercalifragilistic word\", 6) == [\"a\", \"supercalifragilistic\", \"word\"]\n    assert wrap(\"\", 5) == []\n    assert wrap(\"  spaced   out  \", 20) == [\"spaced out\"]")

### Exercise: Digital root

The **digital root** of a number is what you get by adding up its digits, then adding up the digits of that, and so on until one digit is left: 9875 gives 29, then 11, then 2. Write the plan in pseudocode as a comment first, then `digital_root(n)` for whole numbers from 0 up.

In [ ]:
# Plan:
#

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert [digital_root(n) for n in (9875, 0, 7, 10, 999999999999)] == [2, 0, 7, 1, 9]")

### Bug hunt: A step lost in translation

The plan below is correct, but the translation isn't: one step was translated wrongly. Compare them line by line and fix the code.

```
to count how many times each letter appears, ignoring case and anything that isn't a letter:
    make an empty tally
    for each character in the text, in lower case:
        if it's a letter, add one to its tally
    the answer is the tally
```

In [ ]:
def letter_tally(text):
    tally = {}
    for ch in text:
        if ch.isalpha():
            tally[ch] = tally.get(ch, 0) + 1
    return tally

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert letter_tally(\"Aa b!\") == {\"a\": 2, \"b\": 1}\n    assert letter_tally(\"\") == {}")

### Puzzle: From the binary plan

Arrange the lines to translate the corrected binary plan. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Desk-check these

Desk-check each plan on the given input and write the answer.

In [ ]:
# Plan A, with n = 13:
#   set count to 0
#   while n is more than 1:
#       if n is even, halve it, otherwise set it to 3 n + 1
#       add 1 to count
#   the answer is count
a = ___
# Plan B, with the list [3, 8, 2, 9, 4]:
#   set best to the first item, and second to nothing
#   for each item after the first:
#       if it's bigger than best, second becomes best and best becomes the item
#       otherwise, if second is nothing or it's bigger than second, second becomes the item
#   the answer is second
b = ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert a == 9\n    assert b == 8")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: bowling scores

Ten-pin bowling scoring is a classic planning exercise: simple to describe, easy to get wrong. A game has 10 **frames**. In each frame you have up to two rolls to knock down 10 pins.

- A **strike** (all 10 with the first roll) scores 10 plus the next **two rolls**.
- A **spare** (all 10 using both rolls) scores 10 plus the next **one roll**.
- Otherwise the frame scores the pins knocked down.
- In the 10th frame, a strike or spare earns extra rolls (two or one) just to complete its bonus; those extra rolls aren't frames of their own.

Write `score(rolls)` that takes the list of pins knocked down by each roll in a complete game and returns the total. Write the plan in pseudocode first, and desk-check it on `[10, 7, 3, 9, 0, 10, 0, 8, 8, 2, 0, 6, 10, 10, 10, 8, 1]`, which scores 167.

In [ ]:
# Plan:
#

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several games\"\n    assert score([0] * 20) == 0\n    assert score([1] * 20) == 20\n    assert score([5, 5, 3] + [0] * 17) == 16\n    assert score([10, 3, 4] + [0] * 16) == 24\n    assert score([10] * 12) == 300\n    assert score([9, 1] * 10 + [9]) == 190\n    assert score([10, 7, 3, 9, 0, 10, 0, 8, 8, 2, 0, 6, 10, 10, 10, 8, 1]) == 167\n    assert score([0] * 18 + [10, 10, 10]) == 30")

## Warm-up

Here's a recipe someone wrote for finding the smallest number in a list:

```
look at the first number and remember it
go through the rest of the numbers
if a number is smaller than the one you remember, remember it instead
the number you remember at the end is the smallest
```

It isn't Python, but you could write the Python from it without thinking hard, and you could check it on paper with `[5, 3, 8, 1]` in under a minute. That's pseudocode, and you've been reading it all along: the Luhn rule yesterday, the Fisher-Yates shuffle in Week 8.

## What pseudocode is

**Pseudocode** is an algorithm written in structured plain language. There's no official syntax; the rules are only:

- one step per line, in the order they happen;
- indentation shows what's inside a choice or a loop, as in Python;
- choices say **if** ... **otherwise**; repetition says **for each** ..., **while** ..., or **repeat until** ...;
- each step is precise enough that two people would do the same thing, but free of Python's details: no colons, no brackets, no `range(len(...))`.

The right level of detail is: **as much as you need to stop being unsure**. A step like "sort the list" is fine if you know how you'll do it (`sorted`); "find the best route" isn't, until you know what "best" means and how to find it.

Here's a plan and its Python side by side. Hover over (or tap) any line to see where it went:

## Stepwise refinement

For anything bigger than a few lines, write the plan in stages. Start with a handful of big steps, then **refine** each one into smaller steps, until every line is easy to code. The computer scientist Niklaus Wirth called this **stepwise refinement** in 1971, and it's still how experienced programmers attack a new problem.

Problem: *print a month's calendar, given the year, the month and the weekday the month starts on.*

**First version**: the big steps.

```
print the heading
print the day names
print the days in rows of seven
```

**Second version**: refine the hard step. The first two are easy; the third isn't yet.

```
print the heading
print the day names
work out how many days the month has
start the first row with blank spaces for the days before the 1st
for each day of the month:
    add the day to the current row
    if the row has seven days, print it and start a new row
if the last row has anything in it, print it
```

**Third version**: refine anything still unclear. "How many days the month has" needs the month lengths and the leap-year rule. Everything else can go straight into Python.

Every line of the final code traces back to a line of the second version. When something goes wrong, you know which step to look at.

## Desk-checking

A plan can be wrong, and it's far cheaper to find out on paper. **Desk-checking** means following the plan by hand on a small example, writing down the values as you go: exactly the trace tables from Week 3, but before the code exists.

Here's a plan for converting a whole number to binary:

```
to write n in binary:
    start with an empty result
    while n is more than 0:
        put n mod 2 at the end of the result
        set n to n divided by 2, rounded down
    the answer is the result
```

Desk-check it with n = 6:

| n | n mod 2 | result |
|---|---|---|
| 6 | 0 | `0` |
| 3 | 1 | `01` |
| 1 | 1 | `011` |
| 0 | | |

The answer is `011`, but 6 in binary is `110`. The desk check found a bug before a line of code: the digits come out **backwards**, smallest first. The fix is to put each digit at the **front** of the result. And trying n = 0 finds another: the loop never runs, so the answer is empty, where it should be `0`.

## From plan to Python

Once a plan survives a desk check, translating it is mostly mechanical:

| Pseudocode | Python |
|---|---|
| set x to 0 / remember x | `x = 0` |
| for each item in the list | `for item in items:` |
| repeat until done / while ... | `while not done:` / `while ...:` |
| if ... otherwise ... | `if ...:` / `else:` |
| add one to the tally for w | `counts[w] = counts.get(w, 0) + 1` |
| the answer is x | `return x` |
| stop as soon as you find it | `return` inside the loop, or `break` |

Keep the pseudocode as **comments** while you translate, one per step, and delete the ones the code makes obvious. The ones that explain **why** are worth keeping.

## Practice

Each exercise gives a plan in pseudocode. Desk-check it first on a small example, then translate it.

## Quiz

## Daily challenge

## Before you go

Pick a problem you haven't solved yet (tomorrow's Forge, or any exercise from earlier you skipped) and write a two-level plan for it: three to five big steps, then refine the hardest one. Desk-check the plan before you write any code. Tomorrow: decomposition, splitting a big problem into functions that can be built and checked one at a time.